<a href="https://colab.research.google.com/github/Andysimps0n/DL_lecture/blob/main/alex_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import argparse
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# 1. 데이터 다운로드 및 압축 해제
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip -q tiny-imagenet-200.zip

# 2. val 폴더 구조 정리 (PyTorch ImageFolder 호환용)
import os

val_dir = '/content/tiny-imagenet-200/val'
img_dir = os.path.join(val_dir, 'images')
fp = open(os.path.join(val_dir, 'val_annotations.txt'), 'r')
data = fp.readlines()

val_dict = {}
for line in data:
    words = line.split('\t')
    val_dict[words[0]] = words[1]
fp.close()

# 클래스별 폴더 생성 및 이미지 이동
for img, folder in val_dict.items():
    newpath = os.path.join(img_dir, folder)
    if not os.path.exists(newpath):
        os.makedirs(newpath)
    if os.path.exists(os.path.join(img_dir, img)):
        os.rename(os.path.join(img_dir, img), os.path.join(newpath, img))

URL transformed to HTTPS due to an HSTS policy
--2026-03-09 05:54:11--  https://cs231n.stanford.edu/tiny-imagenet-200.zip
Resolving cs231n.stanford.edu (cs231n.stanford.edu)... 171.64.64.64
Connecting to cs231n.stanford.edu (cs231n.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 248100043 (237M) [application/zip]
Saving to: ‘tiny-imagenet-200.zip.2’

tiny-imagenet-200.z 100%[===================>] 236.61M  16.9MB/s    in 15s     

2026-03-09 05:54:27 (15.5 MB/s) - ‘tiny-imagenet-200.zip.2’ saved [248100043/248100043]

replace tiny-imagenet-200/words.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('/content/tiny-imagenet-200/train', transform=transform)
val_dataset = datasets.ImageFolder('/content/tiny-imagenet-200/val/images', transform=transform)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

#얘네는 테스트 데이터를 따로 안 줘서 그냥 검증셋에서 떼어냄
test_loader = val_loader

In [ ]:
from torch.nn.modules import Linear
class AlexNet(nn.Module):
  def __init__(self):
    super(AlexNet, self).__init__()
    self.features = nn.Sequential(

      # Layer 1
     nn.Conv2d(in_channels=3, out_channels=94, kernel_size=11, stride=4, padding=2),
     nn.ReLU(inplace=True),
     nn.MaxPool2d(kernel_size=3, stride=2),

      # Layer 2
     nn.Conv2d(in_channels=94, out_channels=256, kernel_size=5, padding=2),
     nn.ReLU(inplace=True),
     nn.MaxPool2d(kernel_size=3, stride=2),

      # Layer 3
     nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3, padding=1),
     nn.ReLU(inplace=True),

      # Layer 4
     nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3, padding=1),
     nn.ReLU(inplace=True),

      # Layer 5
     nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3, padding=1),
     nn.ReLU(inplace=True),
     nn.MaxPool2d(kernel_size=3, stride=2),


     nn.AdaptiveAvgPool2d((6,6)),
     nn.Flatten(),

     nn.Linear(in_features=9216, out_features=4096),
     nn.ReLU(),
     nn.Dropout(p=0.5),

     nn.Linear(in_features=4096, out_features=4096),
     nn.ReLU(),
     nn.Dropout(p=0.5),

     nn.Linear(in_features=4096, out_features=200)

    )

  def forward(self,x):
    x = self.features(x)
    return x



In [ ]:
model = AlexNet()
model.to('cuda')
loss_function = nn.CrossEntropyLoss()

lr = 0.0001
optimizer = optim.Adam(model.parameters(), lr=lr)

list_train_loss = []
list_val_loss = []
list_acc = []

epoch = 20 # CNN은 성능이 좋아 20회 정도로도 충분합니다.

for i in range(epoch):
    model.train()
    train_loss = 0
    for input_X, true_y in train_loader:

        optimizer.zero_grad()

        input_X = input_X.to('cuda')
        true_y = true_y.to('cuda')

        pred_y = model(input_X)
        loss = loss_function(pred_y, true_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    list_train_loss.append(avg_train_loss)

    # --- [VALIDATION PHASE] ---
    model.eval()
    val_loss = 0
    correct = 0
    with torch.no_grad():
        for input_X, true_y in val_loader:

            input_X, true_y = input_X.to('cuda'), true_y.to('cuda')
            pred_y = model(input_X)
            val_loss += loss_function(pred_y, true_y).item()

            prediction = pred_y.max(1)[1]
            correct += (prediction == true_y).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    acc = (correct / len(val_loader.dataset)) * 100

    list_val_loss.append(avg_val_loss)
    list_acc.append(acc)

    print(f"Epoch [{i+1}/{epoch}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Acc: {acc:.2f}%")

KeyboardInterrupt: 